# Week 9 Post-Class: Solutions

Reference solution for the fine-tune-on-your-own-data exercise.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt
import pathlib
from PIL import Image

keras.utils.set_random_seed(42)

DATA_DIR = './my_data'  # adjust
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def load_split(split):
    """Read my_data/<split>/<class>/*.{jpg,jpeg,png} into NumPy arrays — no TensorFlow."""
    root = pathlib.Path(DATA_DIR) / split
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    images, labels = [], []
    for label_idx, cls in enumerate(classes):
        paths = sorted(p for ext in ('*.jpg', '*.jpeg', '*.png') for p in (root / cls).glob(ext))
        for img_path in paths:
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
            images.append(np.asarray(img, dtype='float32'))
            labels.append(label_idx)
    return np.array(images), np.array(labels), classes

X_train, y_train, class_names = load_split('train')
X_test, y_test, _ = load_split('test')

## Phase 1: Frozen-base

In [ ]:
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=(160, 160, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
# Keep a reference so Phase 2 can REUSE these trained head weights
classifier_head = layers.Dense(len(class_names), activation='softmax')
outputs = classifier_head(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

phase1_history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                           epochs=10, batch_size=BATCH_SIZE, verbose=2)
phase1_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f'Phase 1 test accuracy: {phase1_acc:.4f}')

## Phase 2: Fine-tuning + augmentation

In [ ]:
# Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Rebuild with augmentation, REUSING Phase 1's trained head (don't start from random)
data_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

inputs = keras.Input(shape=(160, 160, 3))
x = data_aug(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = classifier_head(x)  # SAME head object as Phase 1 -> trained weights carry over

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

phase2_history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                           epochs=10, batch_size=BATCH_SIZE, verbose=2)
phase2_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f'\nPhase 1: {phase1_acc:.4f}')
print(f'Phase 2: {phase2_acc:.4f}')
print(f'Improvement: {(phase2_acc - phase1_acc) * 100:.2f} percentage points')

## Predictions visualization

In [ ]:
n = min(8, len(X_test))
sample_idx = np.random.RandomState(0).choice(len(X_test), n, replace=False)
predictions = model.predict(X_test[sample_idx], verbose=0)
predicted = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flat:
    ax.axis('off')
for ax, j, i in zip(axes.flat, range(n), sample_idx):
    ax.imshow(X_test[i].astype('uint8'))
    p = class_names[predicted[j]]
    a = class_names[y_test[i]]
    conf = predictions[j][predicted[j]]
    color = 'green' if p == a else 'red'
    ax.set_title(f'P: {p} ({conf:.2f})\nA: {a}', color=color, fontsize=10)
plt.tight_layout(); plt.show()

## Typical reflection answers

1. **Phase 1 vs Phase 2 accuracy:** Typically Phase 1 gets 75-95%, Phase 2 adds another 1-5%. The exact numbers depend on how similar your domain is to ImageNet.

2. **Plateau point:** Most students see returns flatten around 100-200 training images per class. Beyond that, more data helps marginally.

3. **Common failure modes:**
   - Visually similar classes (different dog breeds, similar food types)
   - Unusual angles or lighting
   - Classes far from ImageNet domain (e.g., medical imagery, microscopy)

4. **Next improvements (in priority):**
   - More training data (most impactful)
   - Better augmentation (color jitter, cutout)
   - Different/larger backbone (EfficientNetB1+, if compute allows)
   - Ensemble of multiple models